# Week 3 Exercise — DevOps Incident Report Generator

**By:** Jaymineh  
**API:** Z.AI (GLM) + OpenRouter (open-source models)

A synthetic data generator that creates realistic DevOps incident / post-mortem reports.  
Useful for training runbook LLMs, testing dashboards, onboarding new SREs, or populating demo environments.

**Week 3 skills demonstrated:**
- Using **frontier** (Z.AI GLM) and **open-source** (Llama via OpenRouter) models side by side
- Structured JSON output parsed from model responses
- Gradio UI with model selection, category filtering, and CSV/JSON download
- Comparing output quality across different model providers

In [ ]:
import os
import json
import csv
import io
import tempfile
import gradio as gr
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

zai_api_key = os.getenv("ZAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if zai_api_key:
    print(f"ZAI_API_KEY found, begins: {zai_api_key[:8]}...")
else:
    print("ZAI_API_KEY not found")

if openrouter_api_key:
    print(f"OPENROUTER_API_KEY found, begins: {openrouter_api_key[:8]}...")
else:
    print("OPENROUTER_API_KEY not found")

In [ ]:
# Initialise both clients

zai_client = OpenAI(
    api_key=zai_api_key,
    base_url="https://api.z.ai/api/paas/v4/",
)

openrouter_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
)

In [ ]:
# Model registry — maps friendly names to (client, model_id)

MODELS = {
    "GLM-4.7 (Z.AI)":            (zai_client, "glm-4.7"),
    "GLM-5 (Z.AI)":              (zai_client, "glm-5"),
    "Llama 3.3 70B (OpenRouter)": (openrouter_client, "meta-llama/llama-3.3-70b-instruct"),
    "Qwen3 70B (OpenRouter)":     (openrouter_client, "qwen/qwen3-70b:free"),
}

CATEGORIES = [
    "All (random mix)",
    "Infrastructure Failure",
    "Deployment / Rollback",
    "Security Incident",
    "Performance Degradation",
    "Database Outage",
    "Network / DNS Issue",
    "Third-Party Service Failure",
]

In [ ]:
SYSTEM_PROMPT = """\
You are a synthetic data generator that produces realistic DevOps incident / post-mortem reports.

Each report must be a JSON object with EXACTLY these fields:
- "incident_id": string, format "INC-XXXX" with a random 4-digit number
- "title": string, short descriptive incident title
- "severity": string, one of "SEV1", "SEV2", "SEV3", "SEV4"
- "category": string, the incident category
- "service_affected": string, the service or system impacted
- "detected_at": string, ISO 8601 datetime
- "resolved_at": string, ISO 8601 datetime (after detected_at)
- "root_cause": string, 1-2 sentence root cause analysis
- "timeline": list of objects [{"time": "ISO datetime", "event": "description"}] with 3-5 entries
- "resolution": string, 1-2 sentence description of what fixed it
- "lessons_learned": string, 1-2 sentence takeaway
- "team_lead": string, realistic full name

Rules:
- Generate realistic, varied data. Use different services, teams, causes each time.
- Dates should be in 2024-2025 range.
- Respond ONLY with a JSON array of objects. No markdown fences, no commentary.
"""

In [ ]:
def build_user_prompt(count: int, category: str) -> str:
    cat_instruction = ""
    if category != "All (random mix)":
        cat_instruction = f' All incidents must be in the "{category}" category.'
    return (
        f"Generate exactly {count} unique DevOps incident reports as a JSON array."
        f"{cat_instruction}"
        f" Ensure each incident is distinct with different services, teams, and root causes."
    )

In [ ]:
def parse_json_response(text: str) -> list[dict]:
    """Extract a JSON array from model output, with repair for common LLM JSON mistakes."""
    import re

    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:])
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]

    start = cleaned.find("[")
    end = cleaned.rfind("]") + 1
    if start == -1 or end == 0:
        raise ValueError(f"No JSON array found in response: {cleaned[:200]}...")

    fragment = cleaned[start:end]

    # Attempt 1: parse as-is
    try:
        return json.loads(fragment)
    except json.JSONDecodeError:
        pass

    # Attempt 2: fix trailing commas before } or ]
    fixed = re.sub(r",\s*([}\]])", r"\1", fragment)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass

    # Attempt 3: response was truncated — close any open structures
    truncated = fixed.rstrip()
    if not truncated.endswith("]"):
        last_brace = truncated.rfind("}")
        if last_brace != -1:
            truncated = truncated[:last_brace + 1] + "]"
    try:
        return json.loads(truncated)
    except json.JSONDecodeError as e:
        raise ValueError(f"Could not parse JSON after repair attempts: {e}\nFragment: {fragment[:300]}...")

In [ ]:
def generate_incidents(model_name: str, count: int, category: str) -> list[dict]:
    """Call the selected model and return parsed incident records."""
    client, model_id = MODELS[model_name]
    user_prompt = build_user_prompt(count, category)

    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.9,
        max_tokens=4096,
    )

    raw = response.choices[0].message.content
    return parse_json_response(raw)

In [ ]:
# Quick test — generate 2 incidents with GLM-4.7

sample = generate_incidents("GLM-4.7 (Z.AI)", 2, "All (random mix)")
print(json.dumps(sample, indent=2))

In [ ]:
def flatten_for_table(incidents: list[dict]) -> pd.DataFrame:
    """Flatten incident records into a DataFrame for display and CSV export."""
    rows = []
    for inc in incidents:
        timeline_str = " | ".join(
            f"{e.get('time', '')}: {e.get('event', '')}" for e in inc.get("timeline", [])
        )
        rows.append({
            "incident_id": inc.get("incident_id", ""),
            "title": inc.get("title", ""),
            "severity": inc.get("severity", ""),
            "category": inc.get("category", ""),
            "service_affected": inc.get("service_affected", ""),
            "detected_at": inc.get("detected_at", ""),
            "resolved_at": inc.get("resolved_at", ""),
            "root_cause": inc.get("root_cause", ""),
            "timeline": timeline_str,
            "resolution": inc.get("resolution", ""),
            "lessons_learned": inc.get("lessons_learned", ""),
            "team_lead": inc.get("team_lead", ""),
        })
    return pd.DataFrame(rows)

In [ ]:
def generate_and_display(model_name, count, category):
    """Gradio callback — generates incidents and returns table + download files."""
    count = int(count)
    try:
        incidents = generate_incidents(model_name, count, category)
    except Exception as e:
        raise gr.Error(f"Generation failed: {e}")

    df = flatten_for_table(incidents)

    # Write CSV to a temp file
    csv_path = os.path.join(tempfile.gettempdir(), "incidents.csv")
    df.to_csv(csv_path, index=False)

    # Write JSON to a temp file
    json_path = os.path.join(tempfile.gettempdir(), "incidents.json")
    with open(json_path, "w") as f:
        json.dump(incidents, f, indent=2)

    status = f"Generated {len(incidents)} incidents using **{model_name}**"
    return status, df, csv_path, json_path

In [ ]:
def compare_models(count, category):
    """Generate with two models and return both tables for comparison."""
    count = int(count)
    results = {}
    for name in ["GLM-4.7 (Z.AI)", "Llama 3.3 70B (OpenRouter)"]:
        try:
            incidents = generate_incidents(name, count, category)
            results[name] = flatten_for_table(incidents)
        except Exception as e:
            results[name] = pd.DataFrame([{"error": str(e)}])

    zai_df = results.get("GLM-4.7 (Z.AI)", pd.DataFrame())
    llama_df = results.get("Llama 3.3 70B (OpenRouter)", pd.DataFrame())
    return zai_df, llama_df

In [ ]:
# Gradio UI

with gr.Blocks(theme=gr.themes.Soft(), title="DevOps Incident Report Generator") as demo:
    gr.Markdown("# DevOps Incident Report Generator")
    gr.Markdown(
        "Generate synthetic incident / post-mortem reports using frontier and open-source LLMs. "
        "Useful for training, dashboards, demos, and onboarding."
    )

    with gr.Tabs():

        # ---- Tab 1: Generate ----
        with gr.TabItem("Generate"):
            with gr.Row():
                with gr.Column(scale=1):
                    model_dd = gr.Dropdown(
                        choices=list(MODELS.keys()),
                        value="GLM-4.7 (Z.AI)",
                        label="Model",
                    )
                    category_dd = gr.Dropdown(
                        choices=CATEGORIES,
                        value="All (random mix)",
                        label="Incident Category",
                    )
                    count_slider = gr.Slider(
                        minimum=1, maximum=10, value=3, step=1,
                        label="Number of Incidents",
                    )
                    generate_btn = gr.Button("Generate", variant="primary")

                with gr.Column(scale=3):
                    status_md = gr.Markdown()
                    preview_table = gr.Dataframe(
                        label="Preview",
                        wrap=True,
                        interactive=False,
                    )
                    with gr.Row():
                        csv_download = gr.File(label="Download CSV")
                        json_download = gr.File(label="Download JSON")

            generate_btn.click(
                fn=generate_and_display,
                inputs=[model_dd, count_slider, category_dd],
                outputs=[status_md, preview_table, csv_download, json_download],
            )

        # ---- Tab 2: Compare Models ----
        with gr.TabItem("Compare Models"):
            gr.Markdown(
                "Generate the same request with **GLM-4.7 (Z.AI)** and **Llama 3.3 70B (OpenRouter)** "
                "side by side to compare output quality."
            )
            with gr.Row():
                cmp_category = gr.Dropdown(
                    choices=CATEGORIES,
                    value="All (random mix)",
                    label="Incident Category",
                )
                cmp_count = gr.Slider(
                    minimum=1, maximum=5, value=2, step=1,
                    label="Incidents per Model",
                )
                cmp_btn = gr.Button("Compare", variant="primary")

            gr.Markdown("### GLM-4.7 (Z.AI)")
            zai_table = gr.Dataframe(wrap=True, interactive=False)

            gr.Markdown("### Llama 3.3 70B (OpenRouter)")
            llama_table = gr.Dataframe(wrap=True, interactive=False)

            cmp_btn.click(
                fn=compare_models,
                inputs=[cmp_count, cmp_category],
                outputs=[zai_table, llama_table],
            )

demo.launch()